In [1]:
from huggingface_hub import login
login()

## Access to Web

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("Who is the current president of France?")
print(results)

The president of France is the head of state of France, elected by popular vote for five years. The first officeholder is considered to be Louis-Napoléon Bonaparte, who was elected in 1848 but provoked the 1851 self-coup to later proclaim himself emperor as Napoleon III. His coup, which proved popular as he sought the restoration of universal male suffrage previously abolished by the ... Mélenchon first served as party president before becoming party co-president alongside Martine Billard, a position he held until 2014. As co-president of the Left Party, he joined the electoral coalition of the Left Front before the 2009 European Parliament election in France. Find out all the news of the President Emmanuel Macron on the official website and discover our pages on the history of the French Republic. Yesterday's elections in France gave the victory to Emmanuel Macron, with 65.5% of the votes, transforming him into the youngest president in the history of… Who is currently the president o

## Creating a Custom Tool for Weather Information to Schedule the Fireworks

In [4]:
from langchain_core.tools import Tool
import random

def get_weather_info(location: str) -> str:
    """Fetches dummy weather information for a given location."""
    # Dummy weather data
    weather_conditions = [
        {"condition": "Rainy", "temp_c": 15},
        {"condition": "Clear", "temp_c": 25},
        {"condition": "Windy", "temp_c": 20}
    ]
    # Randomly select a weather condition
    data = random.choice(weather_conditions)
    return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = Tool(
    name="get_weather_info",
    func=get_weather_info,
    description="Fetches dummy weather information for a given location."
)

##  Creating a Hub Stats Tool for Influential AI Builders

In [6]:
from langchain_core.tools import Tool
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = Tool(
    name="get_hub_stats",
    func=get_hub_stats,
    description="Fetches the most downloaded model from a specific author on the Hugging Face Hub."
)

# Example usage
print(hub_stats_tool.invoke("facebook")) # Example: Get the most downloaded model by Facebook

The most downloaded model by facebook is facebook/contriever with 8,181,369 downloads.


##  Integrating Tools with Alfred

In [7]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Generate the chat interface, including the tools
llm = HuggingFaceEndpoint(
    # repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    repo_id="Qwen/Qwen3.8-27B:novita",
)

chat = ChatHuggingFace(llm=llm, verbose=True)
tools = [search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat.bind_tools(tools)

# Generate the AgentState and Agent graph
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {
        "messages": [chat_with_tools.invoke(state["messages"])],
    }

## The graph
builder = StateGraph(AgentState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message requires a tool, route to tools
    # Otherwise, provide a direct response
    tools_condition,
)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

messages = [HumanMessage(content="Who is Facebook and what's their most popular model?")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Facebook** is a technology company (now operating under its parent company **Meta Platforms, Inc.**) that developed the popular social media platform Facebook. It's a major player in social networking, advertising, and increasingly in artificial intelligence and virtual reality.

As for their most popular model on the Hugging Face Hub, **facebook/contriever** is their most downloaded model, with **8,181,369 downloads**. Contriever is a multilingual retrieval model designed for cross-lingual sentence retrieval, capable of matching text across 100+ languages.
